In [ ]:
import pandas as pd
import numpy as np
import re
import html
import difflib # <-- AGGIUNTO: Necessario per la similitudine tra titoli

# ==============================================================================
# FASE 1: CARICAMENTO, PULIZIA STRUTTURALE E DEDUPLICA
# ==============================================================================

print("--- FASE 1: Caricamento e Pulizia Base ---")

# 1) Load
df = pd.read_csv("../data/raw/goodreadss_books.csv")

# 2) Treat empty/whitespace strings as NaN
df = df.replace(r"^\s*$", np.nan, regex=True)

# 3) Build 'year' if not present
def extract_year(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r"(1[0-9]{3}|20[0-9]{2})", str(val))
    return m.group(0) if m else np.nan

if "year" not in df.columns:
    year = pd.Series(np.nan, index=df.index, dtype="object")
    if "publish_date" in df.columns:
        year = df["publish_date"].apply(extract_year)
    if "first_publish_date" in df.columns:
        year2 = df["first_publish_date"].apply(extract_year)
        year = year.fillna(year2)
    df["year"] = year

# 4) Required fields
required_cols = ["genres", "author", "title", "year", "num_pages", "characters", "first_publish_date","publish_date", "num_ratings", "num_reviews", "avg_rating", "description", "series", "language"]
required_cols = [c for c in required_cols if c in df.columns]

# 5) Drop rows ONLY if title is missing
df_kept = df.dropna(subset=["title"]).copy() if "title" in df.columns else df.copy()

# --- Pulizia Avanzata (Generi e Descrizioni) ---

print("--- Esecuzione Pulizia Testuale (Generi e Descrizioni) ---")

garbage_string = "Art,Biography,Business,Children's,Christian,Classics,Comics,Cookbooks,Ebooks,Fantasy,Fiction,Graphic Novels,Historical Fiction,History,Horror,Memoir,Music,Mystery,Nonfiction,Poetry,Psychology,Romance,Science,Science Fiction,Self Help,Sports,Thriller,Travel,Young Adult"

def clean_garbage_string(genre_str):
    if not isinstance(genre_str, str): 
        return genre_str
    return genre_str.replace(garbage_string, "")

def deduplicate_genres(genre_str):
    if not isinstance(genre_str, str):
        return genre_str
    parts = genre_str.split(',')
    parts = [p.strip() for p in parts if p.strip()]
    seen = set()
    unique_parts = []
    for p in parts:
        if p not in seen:
            unique_parts.append(p)
            seen.add(p)
    res = ",".join(unique_parts)
    return res if res else np.nan

if "genres" in df_kept.columns:
    df_kept["genres"] = df_kept["genres"].apply(clean_garbage_string)
    df_kept["genres"] = df_kept["genres"].apply(deduplicate_genres)

def clean_description_duplication(text):
    if not isinstance(text, str) or len(text) < 100:
        return text
    snippet = text[:30]
    repeat_index = text.find(snippet, 50)
    if repeat_index != -1:
        return text[repeat_index:]
    return text

if "description" in df_kept.columns:
    df_kept["description"] = df_kept["description"].apply(clean_description_duplication)

# --- Strategia "Survival of the Fittest" ---

print("--- Esecuzione Deduplica Titoli (Survival of the Fittest) ---")

# Assicuriamoci che num_ratings sia numerico per l'ordinamento
if "num_ratings" in df_kept.columns:
    df_kept["num_ratings"] = pd.to_numeric(df_kept["num_ratings"], errors='coerce').fillna(0)

rows_pre_dedup = len(df_kept)

# --- MODIFICA RICHIESTA: Deduplica Case-Insensitive ---
# Creiamo una versione minuscola temporanea per identificare i duplicati senza distinzione tra maiuscole/minuscole
df_kept['title_lower'] = df_kept['title'].str.lower()
df_kept = df_kept.sort_values(by=["title_lower", "num_ratings"], ascending=[True, False])
df_kept = df_kept.drop_duplicates(subset=["title_lower"], keep="first")
df_kept = df_kept.drop(columns=['title_lower'])

print(f" -> Righe dopo deduplica titoli (case-insensitive): {len(df_kept)} (rimossi {rows_pre_dedup - len(df_kept)})")

# --- NUOVA AGGIUNTA: Deduplica Fuzzy (Edizioni e Riedizioni) ---
print("--- Esecuzione Deduplica Fuzzy (90% Similarity + Regola Serie) ---")
rows_pre_fuzzy = len(df_kept)
# Calcoliamo l'engagement per decidere chi tenere
df_kept['engagement'] = df_kept['num_ratings'] + (pd.to_numeric(df_kept['num_reviews'], errors='coerce').fillna(0))
df_kept['first_author'] = df_kept['author'].str.split(',').str[0].str.strip().str.lower()

to_remove = set()
authors = df_kept['first_author'].unique()

for auth in authors:
    sub_df = df_kept[df_kept['first_author'] == auth]
    if len(sub_df) < 2: continue
    
    indices = sub_df.index.tolist()
    for i in range(len(indices)):
        for j in range(i + 1, len(indices)):
            idx1, idx2 = indices[i], indices[j]
            if idx1 in to_remove or idx2 in to_remove: continue
            
            t1, t2 = str(sub_df.loc[idx1, 'title']).lower(), str(sub_df.loc[idx2, 'title']).lower()
            # Calcolo somiglianza basato sulla lunghezza del titolo più corto
            matcher = difflib.SequenceMatcher(None, t1, t2)
            match = matcher.find_longest_match(0, len(t1), 0, len(t2))
            sim = match.size / min(len(t1), len(t2)) if min(len(t1), len(t2)) > 0 else 0
            
            if sim >= 0.90:
                # Controlliamo la colonna series (NaN o stringa vuota o 'Unknown')
                s1_empty = pd.isna(sub_df.loc[idx1, 'series']) or str(sub_df.loc[idx1, 'series']).lower() in ['', 'nan', 'unknown']
                s2_empty = pd.isna(sub_df.loc[idx2, 'series']) or str(sub_df.loc[idx2, 'series']).lower() in ['', 'nan', 'unknown']
                
                # Se almeno uno dei due ha la serie vuota, procediamo alla rimozione del meno famoso
                if s1_empty or s2_empty:
                    if sub_df.loc[idx1, 'engagement'] >= sub_df.loc[idx2, 'engagement']:
                        to_remove.add(idx2)
                    else:
                        to_remove.add(idx1)

df_kept = df_kept.drop(index=list(to_remove))
df_kept = df_kept.drop(columns=['engagement', 'first_author'])
print(f" -> Righe dopo deduplica fuzzy: {len(df_kept)} (rimosse {rows_pre_fuzzy - len(df_kept)} riedizioni)")

# Riempimento "Unknown" parziale
cols_to_fill_unknown = [c for c in ["genres", "author", "characters", "description"] if c in df_kept.columns]
df_kept[cols_to_fill_unknown] = df_kept[cols_to_fill_unknown].fillna("Unknown")

# Rimozione colonne inutili
columns_to_remove = ["places", "isbn","isbn13", "publish_date", "first_publish_date", "rated_1","rated_2","rated_3","rated_4","rated_5"] 
df_kept = df_kept.drop(columns=[c for c in columns_to_remove if c in df_kept.columns])

# ==============================================================================
# FASE 2: PREPARAZIONE SPECIFICA PER CHATBOT (FILTRI E FIX NUMERICI)
# ==============================================================================

print("\n--- FASE 2: Ottimizzazione per Chatbot ---")

df = df_kept 

# 1. FIX IMPORTANTE: Conversione num_pages in numeri
if 'num_pages' in df.columns:
    df['num_pages'] = pd.to_numeric(df['num_pages'], errors='coerce')

# --- NUOVA AGGIUNTA: Filtro Pagine < 20 ---
if 'num_pages' in df.columns:
    pre_page_filter = len(df)
    df = df[df['num_pages'] >= 20].copy()
    print(f" -> Filtro Pagine: Rimossi {pre_page_filter - len(df)} libri con meno di 20 pagine.")

# 2. FIX AGGIUNTO: Conversione num_reviews in numeri
if 'num_reviews' in df.columns:
    df['num_reviews'] = pd.to_numeric(df['num_reviews'], errors='coerce').fillna(0)

# 3. FILTRO RICHIESTO (Valutazioni e Recensioni)
MIN_RATINGS = 10
MIN_REVIEWS = 20

if 'num_ratings' in df.columns and 'num_reviews' in df.columns:
    initial_len = len(df)
    df = df[ (df['num_ratings'] >= MIN_RATINGS) | (df['num_reviews'] >= MIN_REVIEWS) ].copy()
    print(f" -> Filtro popolarità: Rimossi libri con < {MIN_RATINGS} ratings E < {MIN_REVIEWS} reviews.")
    print(f" -> Righe rimaste: {len(df)}")

# 4. SERIES PARSING
def parse_series(text):
    if pd.isna(text) or text == "Unknown":
        return None, None
    match = re.search(r"^(.*)\s+#(\d+(\.\d+)?(-\d+)?)$", str(text))
    if match:
        return match.group(1).strip(), match.group(2)
    return text, None

if 'series' in df.columns:
    df[['series_name', 'series_vol']] = df['series'].apply(lambda x: pd.Series(parse_series(x)))
    print(" -> Serie parsate.")

# 5. PAGES CHECK
if 'num_pages' in df.columns:
    df.loc[df['num_pages'] <= 0, 'num_pages'] = np.nan

# 6. TEXT CLEANING AVANZATO (HTML + Entità)
def clean_html_advanced(text):
    if not isinstance(text, str):
        return text
    text = html.unescape(text)
    clean_tags = re.compile('<.*?>')
    text = re.sub(clean_tags, '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

if 'description' in df.columns:
    df['description'] = df['description'].apply(clean_html_advanced)
    print(" -> Descrizioni pulite.")

# 7. FILTRO LINGUA (MODIFICA RICHIESTA)
if 'language' in df.columns:
    df['language'] = df['language'].fillna('Unknown')
    
    # Normalizzazione codici comuni
    lang_map = {'eng': 'English', 'en-US': 'English', 'en-GB': 'English', 'en-CA': 'English'}
    df['language'] = df['language'].replace(lang_map)
    
    # Lista delle lingue consentite
    allowed_languages = ['English', 'Unknown', 'Multiple Languages']
    
    pre_lang_filter = len(df)
    df = df[df['language'].isin(allowed_languages)]
    print(f" -> Filtro Lingua applicato. Righe rimaste: {len(df)} (rimosse {pre_lang_filter - len(df)})")

# Salva
output_file = "../data/raw/goodreads_chatbot_cleaned.csv"
df.to_csv(output_file, index=False)
print(f"\nDataset COMPLETO salvato: {output_file}")
print(f"Righe totali finali: {len(df)}")

--- FASE 1: Caricamento e Pulizia Base ---
--- Esecuzione Pulizia Testuale (Generi e Descrizioni) ---
--- Esecuzione Deduplica Titoli (Survival of the Fittest) ---
 -> Righe dopo deduplica titoli: 16906 (rimossi 2603)

--- FASE 2: Ottimizzazione per Chatbot ---
 -> Filtro popolarità: Rimossi libri con < 10 ratings E < 20 reviews.
 -> Righe rimaste: 13173
 -> Serie parsate.
 -> Descrizioni pulite.
 -> Filtro Lingua applicato. Righe rimaste: 12380 (rimosse 793)

Dataset COMPLETO salvato: ../data/raw/goodreads_chatbot_cleaned.csv
Righe totali finali: 12380
